# Olist Customer Experience - business analysis
SQL controls, descriptive analysis, reproducible findings and recommendations. No predictive model.

In [1]:
from pathlib import Path
import sqlite3, json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == '04_Python' else Path.cwd()
con = sqlite3.connect(ROOT / '03_SQL/analysis.sqlite')

In [2]:
sql = (ROOT / '03_SQL/business_analysis.sql').read_text()
import re
for name, query in re.findall(r'-- name: (\w+)\n(.*?);', sql, re.S):
    table = pd.read_sql_query(query, con)
    print('\n'+name+'\n'+table.head(12).to_string(index=False))


portfolio_kpis
 orders  delivered  cancelled  cancellation_rate  late_rate  on_time_rate  delivery_days  processing_days  shipping_days  review_score   freight   item_aov
  99441      96478        625           0.006285   0.067731      0.932269      12.558217         2.853415       9.333302      4.070796 22.823562 137.754076

status_mix
order_status  orders
    approved       2
    canceled     625
     created       5
   delivered   96478
    invoiced     314
  processing     301
     shipped    1107
 unavailable     609

delivery_trend
purchase_month  orders  eligible  late_rate  delivery_days  review_score  cancelled_rate
       2016-09       4         1   1.000000      54.813194      1.000000        0.500000
       2016-10     324       265   0.007547      19.600559      3.518519        0.074074
       2016-12       1         1   0.000000       4.693021      5.000000        0.000000
       2017-01     800       750   0.029333      12.647044      4.046250        0.003750
       201

In [3]:
findings = json.loads((ROOT / '08_Documentation/findings.json').read_text())
print('FINDINGS\n'+'\n'.join(findings['findings']))
print('\nRECOMMENDATIONS\n'+'\n'.join(findings['recommendations']))
con.close()

FINDINGS
99,441 orders; 96,478 delivered; 625 cancelled (0.63%).
96,470 orders had valid delivery and estimated dates; 6.77% were late by calendar date.
Mean review was 4.28 for on-time orders versus 2.26 for late orders; association does not establish causation.
Mean processing time was 2.85 days and mean shipping time was 9.33 days; available-timestamp samples differ.
170 sellers met the priority rule: at least 50 eligible orders and a late rate above the portfolio rate.
99,441 orders had a selected review; overall mean score was 4.07.

RECOMMENDATIONS
Audit high-volume sellers with above-baseline late rates first; verify carrier handoff records and inventory availability before attributing responsibility.
Compare processing and shipping separately on the same complete-timestamp sample before assigning an operational intervention.
Test revised delivery estimates on routes with persistent positive delays; track both promise accuracy and total transit time.
Use matched-route comparison

In [4]:
print((ROOT / '08_Documentation/statistical_test.json').read_text())

{
  "test": "Mann-Whitney U, descriptive association, exploratory",
  "n_a": 89936,
  "n_b": 6534,
  "mean_a": 4.2794876356520195,
  "mean_b": 2.2552800734618916,
  "p_value": 0.0,
  "caution": "Unadjusted for confounding and multiple comparisons. No causal interpretation."
}


In [5]:
correlations = pd.read_csv(ROOT / '08_Documentation/spearman_correlations.csv', index_col=0)
print(correlations.to_string())
# Correlations are descriptive and cannot establish causation.

                 delay_days  review_score   freight       gmv  processing_days  shipping_days  delivery_days
delay_days         1.000000     -0.179029 -0.155216 -0.043485         0.168044       0.234281       0.311613
review_score      -0.179029      1.000000 -0.087012 -0.033048        -0.104321      -0.197228      -0.236507
freight           -0.155216     -0.087012  1.000000  0.469040         0.043206       0.407585       0.381505
gmv               -0.043485     -0.033048  0.469040  1.000000         0.051009       0.084995       0.101691
processing_days    0.168044     -0.104321  0.043206  0.051009         1.000000       0.040777       0.379579
shipping_days      0.234281     -0.197228  0.407585  0.084995         0.040777       1.000000       0.889813
delivery_days      0.311613     -0.236507  0.381505  0.101691         0.379579       0.889813       1.000000


## Interpretation limits
- Calendar dates define on-time delivery because estimated dates are day-level promises. Same-day delivery counts as on-time.
- Latest review response, then creation time and review ID, determines one selected review per order. Raw review history remains unchanged.
- Processing and shipping averages exclude negative timestamp intervals independently; their samples may differ.
- Multi-seller and multi-category orders count once at order grain. Seller/category comparisons use distinct order membership and cannot uniquely assign blame.
- No carrier identifier exists. Shipping bottlenecks are measured by routes and elapsed time, not named-carrier performance.
- Review associations do not prove operational causation; confounding by category, geography, season and seller mix remains.